# OCEI™ v2.3 — Institutional Orbital Compute Underwriting, 2D Sensitivity Matrix & 4-Tab Excel (.xlsx) Generator
Live zero-auth connection to [OrbitCompute.ai](https://orbitcompute.ai) (RFC-8785 + ECDSA P-256 attested, CERN/Zenodo Concept DOI: `10.5281/zenodo.24181722`).

**Generates a 4-sheet Excel workbook (`OCEI_v2.3_Institutional_Audit.xlsx`):**
1. `1_Scenario_Reconciliation`: 20 reference scenarios across Scale (10 MW–10 GW), Hardware/Launch Stacks, Thermal/Orbit/Downlink boundaries, Skeptic stress (0.6x Self-Build, 3.5%/9% Attrition), Live 3-Hub Grids & G3/G5 Storms.
2. `2_Grid_x_Stack_Matrix`: 2D institutional sensitivity matrix (`$/MTok` spread & dispatch signal across `$10–$250/MWh` × 6 orbital stacks).
3. `3_Live_3Hub_Arbitrage`: Real-time ERCOT Texas, PJM Virginia & EPEX Europe wholesale prices vs. orbital breakeven.
4. `4_D1_24h_TimeSeries`: Empirical Cloudflare D1 15-minute snapshots & rolling 30-day covenant availability.

In [ ]:
import hashlib, json, urllib.parse, urllib.request
from datetime import datetime, timezone
import pandas as pd

BASE = 'https://orbitcompute.ai'

def fetch_json(path, params=None):
    qs = ('?' + urllib.parse.urlencode(params)) if params else ''
    req = urllib.request.Request(BASE + path + qs, headers={'User-Agent': 'OCEI-Colab-Audit/2.3'})
    with urllib.request.urlopen(req, timeout=15) as r:
        return json.loads(r.read().decode('utf-8'))

def verify_rfc8785_sha256(payload):
    att = payload.get('attestation', {})
    unsigned = {k: v for k, v in payload.items() if k != 'attestation'}
    canon = json.dumps(unsigned, sort_keys=True, separators=(',', ':'), ensure_ascii=False).encode('utf-8')
    calc = hashlib.sha256(canon).hexdigest()
    return calc == att.get('state_hash_sha256'), att.get('key_id'), calc

live = fetch_json('/v1/economics/tco-comparator')
ok, kid, digest = verify_rfc8785_sha256(live)
print(f'RFC-8785 SHA-256 Verified: {ok} | Key: {kid} | Hash: {digest[:16]}...')

In [ ]:
SCENARIOS = [
    ('A1_Scale_10MW_Pilot_V3', dict(power_mw=10, hw='rubin_nvl72', launch_vehicle='starship_v3', loop_temp_c=80, grid_price_usd_mwh=115)),
    ('A2_Scale_100MW_Canonical_V3', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', loop_temp_c=80, grid_price_usd_mwh=115)),
    ('A3_Scale_1GW_Hyperscale_V3', dict(power_mw=1000, hw='rubin_nvl72', launch_vehicle='starship_v3', loop_temp_c=80, grid_price_usd_mwh=115)),
    ('A4_Scale_10GW_Supercluster_V3', dict(power_mw=10000, hw='rubin_nvl72', launch_vehicle='starship_v3', loop_temp_c=80, grid_price_usd_mwh=115)),
    ('B1_Starship_V4_Block4_200t', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v4', loop_temp_c=80, grid_price_usd_mwh=115)),
    ('B2_Suncatcher_Trillium_NewGlenn', dict(power_mw=100, hw='trillium_tpu_v6e', launch_vehicle='new_glenn_9x4', loop_temp_c=80, grid_price_usd_mwh=115)),
    ('B3_Starcloud2_Blackwell_V3', dict(power_mw=100, hw='blackwell_b200', launch_vehicle='starship_v3', loop_temp_c=80, grid_price_usd_mwh=115)),
    ('B4_Terafab_370K_95C_V4', dict(power_mw=100, hw='terafab_370k', launch_vehicle='starship_v4', loop_temp_c=95, grid_price_usd_mwh=115)),
    ('C1_Thermal_65C_ZeroThrottle', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', loop_temp_c=65, grid_price_usd_mwh=115)),
    ('C2_Thermal_95C_CommercialCliff', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', loop_temp_c=95, grid_price_usd_mwh=115)),
    ('C3_Orbit_320km_VLEO_Beta58', dict(power_mw=100, altitude_km=320, beta_deg=58, hw='rubin_nvl72', launch_vehicle='starship_v3', grid_price_usd_mwh=115)),
    ('C4_Downlink_Multimodal_2e-6', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', comm_intensity_bits_per_flop=2e-6, grid_price_usd_mwh=115)),
    ('D1_Skeptic_Earth_0.6x_SelfBuild', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', earth_capex_factor=0.6, grid_price_usd_mwh=115)),
    ('D2_Skeptic_3.5pct_BurnIn_Attr', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', annual_attrition_pct=3.5, grid_price_usd_mwh=115)),
    ('D3_Skeptic_9.0pct_Stress_Attr', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', annual_attrition_pct=9.0, grid_price_usd_mwh=115)),
    ('E1_Live_Hub_ERCOT_Texas', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', grid_hub='ercot')),
    ('E2_Live_Hub_PJM_Virginia', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', grid_hub='pjm')),
    ('E3_Live_Hub_EPEX_Europe', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', grid_hub='epex')),
    ('E4_Storm_G3_Solar_Event', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', storm_mode='g3_storm', grid_price_usd_mwh=115)),
    ('E5_Storm_G5_Carrington', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', storm_mode='g5_extreme', grid_price_usd_mwh=115))
]

rows = []
for label, params in SCENARIOS:
    d = fetch_json('/v1/economics/tco-comparator', params)
    oc, ic = d['ocei_standard'], d['input_configuration']
    sw, th = d['space_weather_and_young_daly_checkpointing'], d['thermodynamic_radiative_cooling']
    om, op = d['orbital_mechanics_and_launch_mass'], d['optical_oisl_laser_network']
    ec, cr = d['token_economics_and_5yr_tco'], d['credit_underwriting_and_covenants']
    rows.append({
        'Scenario': label, 'OCEI_Spot': oc['ocei_spot_realtime_score'], 'OCEI_Structural': oc['ocei_structural_5yr_score'],
        'Spread_bps': oc['net_orbital_spread_bps'], 'Grid_USD_MWh': ic['terrestrial_grid_price_usd_mwh'],
        'Breakeven_USD_MWh': oc['breakeven_grid_usd_mwh'], 'Dispatch': oc['arbitrage_dispatch_signal'],
        'Mass_kg_kW': om['deployed_mass_kg_per_it_kw'], 'MMOD_Mass_t': om['mmod_shielded_total_constellation_mass_tons'],
        'Launches': om['starship_flights_required_at_125t'], 'Net_Rad_W_m2': th['effective_net_heat_rejection_w_m2'],
        'Net_Avail_pct': sw['net_effective_compute_availability_pct'], 'Orbit_USD_MTok': ec['orbital_cost_per_1m_tokens_usd'],
        'Orbit_WACC_MTok': ec['wacc_adjusted_orbital_lcoc_usd_mtok'], 'Earth_USD_MTok': ec['terrestrial_cost_per_1m_tokens_usd'],
        'Orbit_5yr_MUSD': ec['orbital_5yr_total_tco_musd'], 'Earth_5yr_MUSD': ec['terrestrial_5yr_total_tco_musd'],
        'Savings_5yr_MUSD': ec['five_year_net_tco_savings_in_orbit_musd'], 'Orbit_USD_W': cr['orbital_5yr_tco_usd_per_watt'],
        'Payback_Yrs': cr['implied_orbital_payback_years'], 'OGS_Util_pct': op['ogs_downlink_utilization_pct'],
        'Downlink_Bound': op['downlink_bound'], 'SHA256_Prefix': d['attestation']['state_hash_sha256'][:16]
    })

df_scenarios = pd.DataFrame(rows)
df_scenarios

In [ ]:
# 2D institutional Sensitivity Matrix: Grid Price ($/MWh) x Stack Architecture (5-Yr Net TCO Savings $M / 100 MW)
GRID_PRICES = [10, 25, 45, 80, 115, 180, 250]
STACKS = [
    ('Rubin_V3_Base', dict(hw='rubin_nvl72', launch_vehicle='starship_v3')),
    ('Rubin_V4_200t', dict(hw='rubin_nvl72', launch_vehicle='starship_v4')),
    ('Trillium_NewGlenn', dict(hw='trillium_tpu_v6e', launch_vehicle='new_glenn_9x4')),
    ('Terafab_370K_V4_95C', dict(hw='terafab_370k', launch_vehicle='starship_v4', loop_temp_c=95)),
    ('Rubin_V3_Earth0.6x', dict(hw='rubin_nvl72', launch_vehicle='starship_v3', earth_capex_factor=0.6)),
    ('Rubin_V3_Attr9pct', dict(hw='rubin_nvl72', launch_vehicle='starship_v3', annual_attrition_pct=9.0))
]

matrix_rows = []
for gp in GRID_PRICES:
    r = {'Grid_USD_MWh': gp}
    for s_name, s_params in STACKS:
        d = fetch_json('/v1/economics/tco-comparator', dict(power_mw=100, grid_price_usd_mwh=gp, **s_params))
        ec = d['token_economics_and_5yr_tco']
        r[f'{s_name}_Orb_MTok'] = ec['orbital_cost_per_1m_tokens_usd']
        r[f'{s_name}_Earth_MTok'] = ec['terrestrial_cost_per_1m_tokens_usd']
        r[f'{s_name}_Save_MUSD'] = ec['five_year_net_tco_savings_in_orbit_musd']
    matrix_rows.append(r)

df_matrix = pd.DataFrame(matrix_rows)
grids = fetch_json('/v1/telemetry/live-grids')
hist = fetch_json('/v1/telemetry/history', {'hours': 24, 'limit': 96})
df_grids = pd.DataFrame.from_dict(grids['live_terrestrial_spot_grids'], orient='index').reset_index(names='hub_id')
df_hist = pd.DataFrame(hist.get('snapshots', []))

out_file = f"OCEI_v2.3_Institutional_Audit_{datetime.now(timezone.utc).strftime('%Y-%m-%d')}.xlsx"
with pd.ExcelWriter(out_file) as writer:
    df_scenarios.to_excel(writer, sheet_name='1_Scenario_Reconciliation', index=False)
    df_matrix.to_excel(writer, sheet_name='2_Grid_x_Stack_Matrix', index=False)
    df_grids.to_excel(writer, sheet_name='3_Live_3Hub_Arbitrage', index=False)
    df_hist.to_excel(writer, sheet_name='4_D1_24h_TimeSeries', index=False)

print(f'Generated 4-sheet institutional Excel workbook: {out_file} ({len(df_hist)} D1 snapshots included)')